# EDA inicial da camada raw

Objetivo: mapear estrutura, qualidade, chaves e mascaramento antes da camada `staging`.

O notebook está organizado em duas partes: inventário global e análises por base.


## Perguntas do EDA

- Que arquivos existem e qual o volume de cada um?
- Quais schemas mudam entre anos/tabelas?
- Quais campos podem ser chaves reais de integração?
- Onde há nulos, categorias inesperadas ou duplicidade de chave?
- O que precisa virar regra/teste no dbt?


In [1]:
from pathlib import Path
import csv
import gzip
import re
import unicodedata

import duckdb
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 20)
pd.set_option("display.max_colwidth", 120)

CWD = Path.cwd().resolve()
if (CWD / "data" / "raw").exists():
    ROOT = CWD
elif (CWD.parent / "data" / "raw").exists():
    ROOT = CWD.parent
else:
    raise FileNotFoundError("Não encontrei data/raw. Abra o notebook na raiz do repo ou dentro de notebooks/.")

RAW_DIR = ROOT / "data" / "raw"
SEEDS_DIR = ROOT / "seeds"
DB_PATH = ROOT / "edu_impacto_nem_multimodal.duckdb"

con = duckdb.connect(str(DB_PATH))

In [3]:
saeb_sample_file = RAW_DIR / "saeb_2019" / "saeb_resultados_municipios_2019.csv.gz"

sample_saeb_municipal = con.sql(
    f"""
    select *
    from read_csv_auto('{saeb_sample_file.as_posix()}')
    limit 3
    """
).df()

sample_saeb_municipal

,CO_UF,NO_UF,CO_MUNICIPIO,NO_MUNICIPIO,DEPENDENCIA_ADM,LOCALIZACAO,MEDIA_5_LP,MEDIA_5_MT,MEDIA_9_LP,MEDIA_9_MT,MEDIA_12_LP,MEDIA_12_MT,nivel_0_LP5,nivel_1_LP5,nivel_2_LP5,nivel_3_LP5,nivel_4_LP5,nivel_5_LP5,nivel_6_LP5,nivel_7_LP5,nivel_8_LP5,nivel_9_LP5,nivel_0_MT5,nivel_1_MT5,nivel_2_MT5,nivel_3_MT5,nivel_4_MT5,nivel_5_MT5,nivel_6_MT5,nivel_7_MT5,nivel_8_MT5,nivel_9_MT5,nivel_10_MT5,nivel_0_LP9,nivel_1_LP9,nivel_2_LP9,nivel_3_LP9,nivel_4_LP9,nivel_5_LP9,nivel_6_LP9,nivel_7_LP9,nivel_8_LP9,nivel_0_MT9,nivel_1_MT9,nivel_2_MT9,nivel_3_MT9,nivel_4_MT9,nivel_5_MT9,nivel_6_MT9,nivel_7_MT9,nivel_8_MT9,nivel_9_MT9,nivel_0_LP12,nivel_1_LP12,nivel_2_LP12,nivel_3_LP12,nivel_4_LP12,nivel_5_LP12,nivel_6_LP12,nivel_7_LP12,nivel_8_LP12,nivel_0_MT12,nivel_1_MT12,nivel_2_MT12,nivel_3_MT12,nivel_4_MT12,nivel_5_MT12,nivel_6_MT12,nivel_7_MT12,nivel_8_MT12,nivel_9_MT12,nivel_10_MT12
0,11,Rondônia,1100015,Alta Floresta D'Oeste,Estadual,Total,210.48,233.47,268.37,273.59,278.52,285.14,2.34,7.02,17.32,13.40,20.85,21.03,9.00,6.50,1.84,0.71,0.00,1.45,5.52,13.83,24.08,23.48,13.97,10.35,5.14,2.17,0.0,4.70,8.29,18.62,21.38,27.72,10.77,7.49,1.04,0.0,3.75,8.71,16.94,21.67,21.57,16.06,7.32,3.5,0.48,0.0,10.69,14.36,20.88,21.6,15.9,9.77,6.3,0.5,0.0,12.41,13.63,12.98,22.16,16.69,14.41,5.81,1.9,0.0,0.0,0.0
1,11,Rondônia,1100015,Alta Floresta D'Oeste,Estadual,Urbana,210.48,233.47,268.37,273.59,278.52,285.14,2.34,7.02,17.32,13.40,20.85,21.03,9.00,6.50,1.84,0.71,0.00,1.45,5.52,13.83,24.08,23.48,13.97,10.35,5.14,2.17,0.0,4.70,8.29,18.62,21.38,27.72,10.77,7.49,1.04,0.0,3.75,8.71,16.94,21.67,21.57,16.06,7.32,3.5,0.48,0.0,10.69,14.36,20.88,21.6,15.9,9.77,6.3,0.5,0.0,12.41,13.63,12.98,22.16,16.69,14.41,5.81,1.9,0.0,0.0,0.0
2,11,Rondônia,1100015,Alta Floresta D'Oeste,Municipal,Rural,189.48,203.69,254.80,252.27,NaN,NaN,5.96,15.89,18.21,12.91,30.46,8.94,4.63,2.98,0.00,0.00,2.32,10.60,15.89,28.14,10.60,15.89,5.30,8.28,2.98,0.00,0.0,9.23,13.23,21.23,29.23,17.85,9.23,0.00,0.00,0.0,8.62,20.62,13.23,26.46,13.85,17.23,0.00,0.0,0.00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Funções utilitárias

A lógica abaixo se repete para todas as bases. Ela tenta detectar encoding, delimitador, cabeçalho, família e ano. Isso evita criar uma célula quase igual para cada arquivo.

In [ ]:
def normalize_name(value: str) -> str:
    """
    Normaliza um nome textual para um formato padronizado de comparação.

    A função remove espaços nas extremidades, elimina BOM UTF-8, remove acentos,
    substitui caracteres não alfanuméricos por "_", remove underscores excedentes
    nas extremidades e converte o resultado para minúsculas.

    Essa normalização é útil para comparar nomes de colunas que podem variar em
    caixa, acentuação, espaços, hífens ou outros separadores.

    Args:
        value (str): Valor textual a ser normalizado.

    Returns:
        str: Nome normalizado em minúsculas, sem acentos e com separadores
        padronizados por "_".
    """

    value = str(value).strip().replace("\ufeff", "")
    value = unicodedata.normalize("NFKD", value)
    value = "".join(ch for ch in value if not unicodedata.combining(ch))
    value = re.sub(r"[^0-9a-zA-Z]+", "_", value).strip("_").lower()
    return value


def detect_year(path: Path):
    """
    Extrai o primeiro ano no formato 20XX encontrado no caminho do arquivo.

    A busca é feita sobre o caminho completo convertido para string, não apenas
    sobre o nome do arquivo.

    Args:
        path (Path): Caminho do arquivo.

    Returns:
        int | None: Ano encontrado como inteiro, ou None caso nenhum ano seja identificado.
    """
    match = re.search(r"(20\d{2})", str(path))
    return int(match.group(1)) if match else None


def detect_family(path: Path) -> str:
    """
    Identifica a família/base de dados a partir do caminho ou nome do arquivo.

    A classificação é feita por regras simples baseadas em palavras-chave no
    caminho completo e no nome do arquivo.

    Args:
        path (Path): Caminho do arquivo.

    Returns:
        str: Família identificada. Pode ser:
            - "saeb"
            - "censo_escolar"
            - "pib"
            - "ibge"
            - "outros"
    """

    text = str(path).lower()
    name = path.name.lower()
    if "saeb" in text:
        return "saeb"
    if "censo_escolar" in text or name.startswith("tabela_"):
        return "censo_escolar"
    if "pib" in text:
        return "pib"
    if "ibge" in text or "alfabetizacao" in text:
        return "ibge"
    return "outros"


def open_text(path: Path, encoding: str):
    """
    Abre um arquivo de texto comum ou compactado em gzip.

    Se o arquivo tiver sufixo ".gz", ele é aberto com gzip em modo texto.
    Caso contrário, é aberto com a função open padrão.

    Args:
        path (Path): Caminho do arquivo.
        encoding (str): Encoding usado para leitura do conteúdo textual.

    Returns:
        TextIO: Objeto de arquivo aberto em modo leitura de texto.
    """

    if path.suffix == ".gz":
        return gzip.open(path, "rt", encoding=encoding, newline="")
    return open(path, "r", encoding=encoding, newline="")


def detect_encoding(path: Path, candidates=("utf-8-sig", "utf-8", "cp1252", "ISO-8859-1")) -> str:
    """
    Detecta o encoding provável de um arquivo testando uma lista de candidatos.

    A função tenta abrir e ler uma amostra inicial do arquivo com cada encoding.
    O primeiro encoding que não gerar UnicodeDecodeError é retornado.

    Args:
        path (Path): Caminho do arquivo.
        candidates (tuple[str, ...]): Encodings candidatos, na ordem de tentativa.

    Returns:
        str: Primeiro encoding que conseguiu ler a amostra sem erro. Se todos
        falharem, retorna o último candidato da lista.
    """

    for encoding in candidates:
        try:
            with open_text(path, encoding) as f:
                f.read(8192)
            return encoding
        except UnicodeDecodeError:
            continue
    return candidates[-1]


def detect_delimiter(path: Path, encoding: str, delimiters=(";", ",", "\t", "|")) -> str:
    """
    Detecta o delimitador provável de um arquivo CSV ou CSV compactado.

    Primeiro utiliza csv.Sniffer para tentar inferir o delimitador a partir
    de uma amostra do arquivo. Se a inferência falhar, usa uma estratégia de
    fallback baseada na contagem dos delimitadores na primeira linha.

    Args:
        path (Path): Caminho do arquivo.
        encoding (str): Encoding usado para leitura do arquivo.
        delimiters (tuple[str, ...]): Delimitadores candidatos.

    Returns:
        str: Delimitador detectado.
    """

    with open_text(path, encoding) as f:
        sample = f.read(32768)
    try:
        return csv.Sniffer().sniff(sample, delimiters=delimiters).delimiter
    except csv.Error:
        first_line = sample.splitlines()[0] if sample else ""
        counts = {delimiter: first_line.count(delimiter) for delimiter in delimiters}
        return max(counts, key=counts.get)


def read_header(path: Path, encoding: str, delimiter: str) -> list[str]:
    """
    Lê o cabeçalho de um arquivo CSV.

    A função considera apenas a primeira linha do arquivo e utiliza o delimitador
    informado para separar os nomes das colunas.

    Args:
        path (Path): Caminho do arquivo.
        encoding (str): Encoding usado para leitura do arquivo.
        delimiter (str): Delimitador usado no CSV.

    Returns:
        list[str]: Lista com os nomes das colunas do arquivo.
    """
    with open_text(path, encoding) as f:
        return next(csv.reader(f, delimiter=delimiter))


def duckdb_read_expr(path: Path, delimiter: str, encoding: str) -> str:
    """
    Monta uma expressão SQL do DuckDB para leitura de um arquivo CSV.

    A expressão gerada usa read_csv_auto com cabeçalho, delimitador detectado,
    leitura de todas as colunas como VARCHAR e tolerância a erros de parsing.
    Essa abordagem é útil para uma camada inicial de EDA ou raw data, evitando
    inferência automática instável de tipos.

    Args:
        path (Path): Caminho do arquivo CSV ou CSV compactado.
        delimiter (str): Delimitador usado no arquivo.
        encoding (str): Encoding detectado para o arquivo.

    Returns:
        str: Expressão SQL para ser usada em consultas DuckDB.
    """

    # all_varchar evita inferencia instavel no EDA inicial; casts entram depois no staging.
    safe_path = str(path).replace("'", "''")
    safe_delim = delimiter.replace("'", "''")
    duckdb_encoding = "latin-1" if encoding.lower() in {"iso-8859-1", "latin-1"} else "utf-8"
    return (
        "read_csv_auto("
        f"'{safe_path}', "
        "header=true, "
        f"delim='{safe_delim}', "
        "all_varchar=true, "
        "ignore_errors=true, "
        f"encoding='{duckdb_encoding}'"
        ")"
    )


def sql_identifier(name: str) -> str:
    """
    Escapa um nome para uso seguro como identificador SQL.

    A função envolve o nome com aspas duplas e duplica aspas internas,
    evitando quebra de sintaxe ao usar nomes de colunas ou tabelas com espaços,
    acentos, caracteres especiais ou palavras reservadas.

    Args:
        name (str): Nome do identificador SQL.

    Returns:
        str: Identificador SQL escapado com aspas duplas.
    """
    return '"' + name.replace('"', '""') + '"'


def read_csv_sample(row, nrows=10, usecols=None):
    """
    Lê uma pequena amostra de um arquivo CSV usando Pandas.

    A função usa as informações de caminho, delimitador e encoding presentes
    em uma linha de metadados, normalmente vinda de um catálogo ou inventário
    de arquivos.

    Args:
        row: Objeto indexável contendo pelo menos as chaves "path",
            "delimiter" e "encoding".
        nrows (int): Número máximo de linhas a serem lidas.
        usecols (list[str] | None): Lista opcional de colunas a serem lidas.

    Returns:
        pandas.DataFrame: Amostra do arquivo CSV com todas as colunas como string.
    """

    return pd.read_csv(
        row["path"],
        sep=row["delimiter"],
        encoding=row["encoding"],
        compression="infer",
        dtype=str,
        nrows=nrows,
        usecols=usecols,
        low_memory=False,
    )


def iter_csv_chunks(row, columns=None, chunksize=200_000):
    """
    Lê um arquivo CSV em partes usando Pandas.

    Essa função é útil para processar arquivos grandes sem carregar todo o
    conteúdo em memória. Cada pedaço retornado pelo iterador é um DataFrame.

    Args:
        row: Objeto indexável contendo pelo menos as chaves "path",
            "delimiter" e "encoding".
        columns (list[str] | None): Lista opcional de colunas a serem lidas.
        chunksize (int): Quantidade de linhas por chunk.

    Returns:
        TextFileReader: Iterador de DataFrames, em que cada DataFrame contém
        até chunksize linhas.
    """

    return pd.read_csv(
        row["path"],
        sep=row["delimiter"],
        encoding=row["encoding"],
        compression="infer",
        dtype=str,
        usecols=columns,
        chunksize=chunksize,
        keep_default_na=False,
        low_memory=False,
    )


def list_csv_files(include_seeds=False) -> list[Path]:
    """
    Lista arquivos CSV disponíveis nos diretórios de dados brutos e, opcionalmente, seeds.

    A função busca arquivos com extensão ".csv" e ".csv.gz" dentro de RAW_DIR.
    Quando include_seeds=True, também inclui arquivos ".csv" diretamente dentro
    de SEEDS_DIR.

    Args:
        include_seeds (bool): Indica se os arquivos CSV do diretório SEEDS_DIR
        também devem ser incluídos.

    Returns:
        list[Path]: Lista ordenada de caminhos para arquivos CSV encontrados.
    """

    files = sorted(RAW_DIR.rglob("*.csv")) + sorted(RAW_DIR.rglob("*.csv.gz"))
    if include_seeds:
        files += sorted(SEEDS_DIR.glob("*.csv"))
    return files

## Inventário dos arquivos raw

Esta célula cria um catálogo inicial com caminho, família, ano, encoding, delimitador, quantidade de colunas e tamanho. Se alguma base aparecer com `n_cols = 1`, normalmente é sinal de delimitador errado; aqui isso já deve ser detectado automaticamente.

In [3]:
catalog_rows = []

for path in list_csv_files(include_seeds=False):
    if path.name == ".gitkeep":
        continue
    encoding = detect_encoding(path)
    delimiter = detect_delimiter(path, encoding)
    header = read_header(path, encoding, delimiter)
    catalog_rows.append(
        {
            "file": path.relative_to(ROOT).as_posix(),
            "path": path,
            "family": detect_family(path),
            "year": detect_year(path),
            "encoding": encoding,
            "delimiter": delimiter,
            "n_cols": len(header),
            "size_mb": path.stat().st_size / 1024 / 1024,
            "columns": header,
            "columns_norm": [normalize_name(col) for col in header],
        }
    )

catalog = pd.DataFrame(catalog_rows).sort_values(["family", "year", "file"]).reset_index(drop=True)
catalog[["family", "year", "file", "encoding", "delimiter", "n_cols", "size_mb"]]

,family,year,file,encoding,delimiter,n_cols,size_mb
0,censo_escolar,2019,data/raw/censo_escolar_2019/microdados_ed_basica_2019.csv.gz,cp1252,;,370,26.923068
1,censo_escolar,2020,data/raw/censo_escolar_2020/microdados_ed_basica_2020.csv.gz,cp1252,;,370,26.785557
2,censo_escolar,2021,data/raw/censo_escolar_2021/microdados_ed_basica_2021.csv.gz,cp1252,;,370,26.710618
3,censo_escolar,2022,data/raw/censo_escolar_2022/microdados_ed_basica_2022.csv.gz,cp1252,;,385,27.616155
4,censo_escolar,2023,data/raw/censo_escolar_2023/microdados_ed_basica_2023.csv.gz,cp1252,;,408,32.685275
5,censo_escolar,2023,data/raw/censo_escolar_2023/suplemento_cursos_tecnicos_2023.csv.gz,cp1252,;,29,0.430531
6,censo_escolar,2024,data/raw/censo_escolar_2024/microdados_ed_basica_2024.csv.gz,cp1252,;,426,31.016352
7,censo_escolar,2024,data/raw/censo_escolar_2024/suplemento_cursos_tecnicos_2024.csv.gz,cp1252,;,29,0.507195
8,censo_escolar,2025,data/raw/censo_escolar_2025/Tabela_Curso_Tecnico_2025.csv.gz,cp1252,;,20,0.307333
9,censo_escolar,2025,data/raw/censo_escolar_2025/Tabela_Docente_2025.csv.gz,utf-8-sig,;,156,8.565481


## Contagem de linhas

Esta etapa pode demorar alguns minutos porque precisa varrer arquivos grandes. Ela não carrega tudo em memória.

In [ ]:
def count_rows(row) -> int:
    """
    Conta o número de linhas de dados de um arquivo CSV.

    A função abre o arquivo usando o encoding detectado previamente e desconta
    uma linha referente ao cabeçalho. Arquivos vazios ou contendo apenas
    cabeçalho retornam 0.

    A leitura é feita em modo texto, em vez de DuckDB, para evitar dependência
    do suporte de encoding da versão local do DuckDB, especialmente em bases
    raw com ISO-8859-1 ou encodings similares.

    Args:
        row: Linha de metadados do catálogo contendo pelo menos as chaves
            "path" e "encoding".

    Returns:
        int: Quantidade de linhas de dados do arquivo, sem contar o cabeçalho.
    """

    # Usamos leitura texto aqui porque algumas bases raw estão em ISO-8859-1.
    # Isso evita depender do suporte de encoding da versão local do DuckDB.
    with open_text(row["path"], row["encoding"]) as f:
        return max(sum(1 for _ in f) - 1, 0)


catalog["n_rows"] = [count_rows(row) for _, row in catalog.iterrows()]
catalog[["family", "year", "file", "n_rows", "n_cols", "size_mb"]].sort_values("size_mb", ascending=False)

,family,year,file,n_rows,n_cols,size_mb
4,censo_escolar,2023,data/raw/censo_escolar_2023/microdados_ed_basica_2023.csv.gz,217625,408,32.685275
6,censo_escolar,2024,data/raw/censo_escolar_2024/microdados_ed_basica_2024.csv.gz,215545,426,31.016352
3,censo_escolar,2022,data/raw/censo_escolar_2022/microdados_ed_basica_2022.csv.gz,224649,385,27.616155
0,censo_escolar,2019,data/raw/censo_escolar_2019/microdados_ed_basica_2019.csv.gz,228521,370,26.923068
1,censo_escolar,2020,data/raw/censo_escolar_2020/microdados_ed_basica_2020.csv.gz,224229,370,26.785557
2,censo_escolar,2021,data/raw/censo_escolar_2021/microdados_ed_basica_2021.csv.gz,221140,370,26.710618
10,censo_escolar,2025,data/raw/censo_escolar_2025/Tabela_Escola_2025.csv.gz,214192,302,20.279098
12,censo_escolar,2025,data/raw/censo_escolar_2025/Tabela_Matricula_2025.csv.gz,178766,237,13.863099
9,censo_escolar,2025,data/raw/censo_escolar_2025/Tabela_Docente_2025.csv.gz,178772,156,8.565481
13,censo_escolar,2025,data/raw/censo_escolar_2025/Tabela_Turma_2025.csv.gz,178772,190,6.153510


## Catálogo de schemas

Uma linha por coluna de cada arquivo. Use este dataframe para procurar campos por padrão (`municipio`, `entidade`, `mat`, `medio`, `saeb`, etc.).

In [5]:
schema_rows = []

for _, row in catalog.iterrows():
    for position, column in enumerate(row["columns"], start=1):
        schema_rows.append(
            {
                "family": row["family"],
                "year": row["year"],
                "file": row["file"],
                "position": position,
                "column": column,
                "column_norm": normalize_name(column),
            }
        )

schema_catalog = pd.DataFrame(schema_rows)
schema_catalog.head(30)

,family,year,file,position,column,column_norm
0,censo_escolar,2019,data/raw/censo_escolar_2019/microdados_ed_basica_2019.csv.gz,1,NU_ANO_CENSO,nu_ano_censo
1,censo_escolar,2019,data/raw/censo_escolar_2019/microdados_ed_basica_2019.csv.gz,2,NO_REGIAO,no_regiao
2,censo_escolar,2019,data/raw/censo_escolar_2019/microdados_ed_basica_2019.csv.gz,3,CO_REGIAO,co_regiao
3,censo_escolar,2019,data/raw/censo_escolar_2019/microdados_ed_basica_2019.csv.gz,4,NO_UF,no_uf
4,censo_escolar,2019,data/raw/censo_escolar_2019/microdados_ed_basica_2019.csv.gz,5,SG_UF,sg_uf
...,...,...,...,...,...,...
25,censo_escolar,2019,data/raw/censo_escolar_2019/microdados_ed_basica_2019.csv.gz,26,NU_TELEFONE,nu_telefone
26,censo_escolar,2019,data/raw/censo_escolar_2019/microdados_ed_basica_2019.csv.gz,27,TP_SITUACAO_FUNCIONAMENTO,tp_situacao_funcionamento
27,censo_escolar,2019,data/raw/censo_escolar_2019/microdados_ed_basica_2019.csv.gz,28,CO_ORGAO_REGIONAL,co_orgao_regional
28,censo_escolar,2019,data/raw/censo_escolar_2019/microdados_ed_basica_2019.csv.gz,29,DT_ANO_LETIVO_INICIO,dt_ano_letivo_inicio


In [20]:
schema_catalog[
    schema_catalog["column_norm"].str.contains("municipio", na=False)
]

,family,year,file,position,column,column_norm
6,censo_escolar,2019,data/raw/censo_escolar_2019/microdados_ed_basica_2019.csv.gz,7,NO_MUNICIPIO,no_municipio
7,censo_escolar,2019,data/raw/censo_escolar_2019/microdados_ed_basica_2019.csv.gz,8,CO_MUNICIPIO,co_municipio
376,censo_escolar,2020,data/raw/censo_escolar_2020/microdados_ed_basica_2020.csv.gz,7,NO_MUNICIPIO,no_municipio
377,censo_escolar,2020,data/raw/censo_escolar_2020/microdados_ed_basica_2020.csv.gz,8,CO_MUNICIPIO,co_municipio
746,censo_escolar,2021,data/raw/censo_escolar_2021/microdados_ed_basica_2021.csv.gz,7,NO_MUNICIPIO,no_municipio
...,...,...,...,...,...,...
3366,saeb,2019,data/raw/saeb_2019/saeb_resultados_municipios_2019.csv.gz,4,NO_MUNICIPIO,no_municipio
3438,saeb,2021,data/raw/saeb_2021/saeb_resultados_municipios_2021.csv.gz,4,CO_MUNICIPIO,co_municipio
3439,saeb,2021,data/raw/saeb_2021/saeb_resultados_municipios_2021.csv.gz,5,NO_MUNICIPIO,no_municipio
3551,saeb,2023,data/raw/saeb_2023/saeb_resultados_municipios_2023.csv.gz,4,CO_MUNICIPIO,co_municipio


In [23]:
schema_catalog.groupby(["family", "year"]).size()

family         year
censo_escolar  2019    370
               2020    370
               2021    370
               2022    385
               2023    437
               2024    455
               2025    970
ibge           2022      6
saeb           2019     72
               2021    113
               2023    113
dtype: int64

In [ ]:
comparacao_posicoes = (
    schema_catalog
    .groupby(["family", "position"])["column_norm"]
    .nunique()
    .reset_index(name="qtd_nomes_diferentes")
    .sort_values(["family", "position"])
)

comparacao_posicoes.head(30)

,family,position,qtd_nomes_diferentes
0,censo_escolar,1,1
1,censo_escolar,2,2
2,censo_escolar,3,6
3,censo_escolar,4,6
4,censo_escolar,5,6
...,...,...,...
25,censo_escolar,26,9
26,censo_escolar,27,9
27,censo_escolar,28,9
28,censo_escolar,29,9


In [ ]:
# Procura termos relevantes para decidir quais colunas entram na staging.
TERMS = "municipio|entidade|uf|ano|mat|medio|prof|doc|tur|gest|curso|saeb|media|nivel|dependencia|localizacao"

schema_catalog[
    schema_catalog["column_norm"].str.contains(TERMS, regex=True, na=False)
].sort_values(["family", "year", "file", "position"]).head(200)

,family,year,file,position,column,column_norm
0,censo_escolar,2019,data/raw/censo_escolar_2019/microdados_ed_basica_2019.csv.gz,1,NU_ANO_CENSO,nu_ano_censo
3,censo_escolar,2019,data/raw/censo_escolar_2019/microdados_ed_basica_2019.csv.gz,4,NO_UF,no_uf
4,censo_escolar,2019,data/raw/censo_escolar_2019/microdados_ed_basica_2019.csv.gz,5,SG_UF,sg_uf
5,censo_escolar,2019,data/raw/censo_escolar_2019/microdados_ed_basica_2019.csv.gz,6,CO_UF,co_uf
6,censo_escolar,2019,data/raw/censo_escolar_2019/microdados_ed_basica_2019.csv.gz,7,NO_MUNICIPIO,no_municipio
...,...,...,...,...,...,...
592,censo_escolar,2020,data/raw/censo_escolar_2020/microdados_ed_basica_2020.csv.gz,223,QT_PROF_MONITORES,qt_prof_monitores
593,censo_escolar,2020,data/raw/censo_escolar_2020/microdados_ed_basica_2020.csv.gz,224,IN_PROF_GESTAO,in_prof_gestao
594,censo_escolar,2020,data/raw/censo_escolar_2020/microdados_ed_basica_2020.csv.gz,225,QT_PROF_GESTAO,qt_prof_gestao
595,censo_escolar,2020,data/raw/censo_escolar_2020/microdados_ed_basica_2020.csv.gz,226,IN_PROF_ASSIST_SOCIAL,in_prof_assist_social


## Roteiro por base

As células abaixo reaproveitam as mesmas funções para cada família. A visão estrutural cobre todas as bases; perfis mais pesados ficam configurados por base para não deixar o notebook lento ou ruidoso.

In [7]:
from IPython.display import Markdown, display


def show(title, obj):
    display(Markdown(f"**{title}**"))
    display(obj)


def build_catalog_for_files(files: list[Path], source_layer: str) -> pd.DataFrame:
    rows = []
    for path in files:
        encoding = detect_encoding(path)
        delimiter = detect_delimiter(path, encoding)
        header = read_header(path, encoding, delimiter)
        rows.append(
            {
                "source_layer": source_layer,
                "file": path.relative_to(ROOT).as_posix(),
                "path": path,
                "family": detect_family(path),
                "year": detect_year(path),
                "encoding": encoding,
                "delimiter": delimiter,
                "n_cols": len(header),
                "size_mb": path.stat().st_size / 1024 / 1024,
                "columns": header,
                "columns_norm": [normalize_name(col) for col in header],
            }
        )
    return pd.DataFrame(rows)


seed_catalog = build_catalog_for_files(sorted(SEEDS_DIR.glob("*.csv")), source_layer="seed")
raw_catalog = catalog.assign(source_layer="raw")
all_catalog = pd.concat([raw_catalog, seed_catalog], ignore_index=True, sort=False)


def schema_from_catalog(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in df.iterrows():
        for position, column in enumerate(row["columns"], start=1):
            rows.append(
                {
                    "source_layer": row.get("source_layer", "raw"),
                    "family": row["family"],
                    "year": row["year"],
                    "file": row["file"],
                    "position": position,
                    "column": column,
                    "column_norm": normalize_name(column),
                }
            )
    return pd.DataFrame(rows)


all_schema_catalog = schema_from_catalog(all_catalog)

BASE_GROUPS = {
    "saeb": r"saeb_20(?:19|21|23)/saeb_resultados",
    "censo_escolar_2019_2024": r"censo_escolar_20(?:19|20|21|22|23|24)/microdados",
    "censo_escolar_2025": r"censo_escolar_2025/Tabela_",
    "ibge_raw": r"br_ibge_censo_2022_alfabetizacao",
    "seeds": r"^seeds/",
}


def files_for_group(group: str, include_seeds=True) -> pd.DataFrame:
    pattern = BASE_GROUPS[group]
    source = all_catalog if include_seeds else raw_catalog
    return source[source["file"].str.contains(pattern, regex=True, na=False)].copy()


def inventory_by_group() -> pd.DataFrame:
    rows = []
    for group in BASE_GROUPS:
        files = files_for_group(group)
        rows.append(
            {
                "base": group,
                "n_files": len(files),
                "years": sorted(files["year"].dropna().astype(int).unique().tolist()),
                "total_size_mb": files["size_mb"].sum(),
                "min_cols": files["n_cols"].min() if len(files) else None,
                "max_cols": files["n_cols"].max() if len(files) else None,
            }
        )
    return pd.DataFrame(rows)


inventory_by_group()

,base,n_files,years,total_size_mb,min_cols,max_cols
0,saeb,3,"[2019, 2021, 2023]",7.532529,72,113
1,censo_escolar_2019_2024,6,"[2019, 2020, 2021, 2022, 2023, 2024]",171.737024,370,426
2,censo_escolar_2025,6,[2025],50.949284,20,302
3,ibge_raw,1,[2022],4.097044,6,6
4,seeds,3,[2022],1.935221,7,27


In [8]:
def compare_columns(files_filter: str, schema_df=all_schema_catalog):
    df = schema_df[schema_df["file"].str.contains(files_filter, regex=True, na=False)].copy()
    presence = (
        df.assign(value=1)
        .pivot_table(index="column_norm", columns="file", values="value", aggfunc="max", fill_value=0)
        .reset_index()
    )
    if presence.empty:
        return presence
    presence["n_files"] = presence.drop(columns="column_norm").sum(axis=1)
    return presence.sort_values(["n_files", "column_norm"], ascending=[True, True])


def get_catalog_row(file_contains: str, source=all_catalog):
    matches = source[source["file"].str.contains(file_contains, regex=False, na=False)]
    if matches.empty:
        raise ValueError(f"Nenhum arquivo encontrado contendo: {file_contains}")
    if len(matches) > 1:
        display(matches[["file", "family", "year", "n_cols", "size_mb"]])
        raise ValueError("Filtro encontrou mais de um arquivo. Refine o texto.")
    return matches.iloc[0]


def sample_file(file_contains: str, limit=10, columns=None):
    row = get_catalog_row(file_contains)
    usecols = existing_columns(row, columns) if columns else None
    return read_csv_sample(row, nrows=int(limit), usecols=usecols)


def existing_columns(row, columns):
    return [col for col in columns if col in row["columns"]]


def profile_columns(file_contains: str, columns=None, sample_limit=None, chunksize=200_000):
    row = get_catalog_row(file_contains)
    columns = existing_columns(row, columns or row["columns"])
    rows_seen = 0
    stats = {col: {"n_missing": 0, "distinct": set(), "min_value": None, "max_value": None} for col in columns}

    for chunk in iter_csv_chunks(row, columns=columns, chunksize=chunksize):
        if sample_limit is not None:
            remaining = int(sample_limit) - rows_seen
            if remaining <= 0:
                break
            chunk = chunk.head(remaining)
        rows_seen += len(chunk)
        for col in columns:
            series = chunk[col].astype(str).str.strip()
            non_missing = series[series != ""]
            stats[col]["n_missing"] += int((series == "").sum())
            stats[col]["distinct"].update(non_missing.dropna().unique().tolist())
            if not non_missing.empty:
                stats[col]["min_value"] = non_missing.min() if stats[col]["min_value"] is None else min(stats[col]["min_value"], non_missing.min())
                stats[col]["max_value"] = non_missing.max() if stats[col]["max_value"] is None else max(stats[col]["max_value"], non_missing.max())

    return pd.DataFrame(
        [
            {
                "column": col,
                "n_rows": rows_seen,
                "n_missing": stats[col]["n_missing"],
                "missing_pct": stats[col]["n_missing"] / rows_seen if rows_seen else None,
                "n_distinct": len(stats[col]["distinct"]),
                "min_value": stats[col]["min_value"],
                "max_value": stats[col]["max_value"],
            }
            for col in columns
        ]
    )


def value_counts(file_contains: str, column: str, limit=30, chunksize=200_000):
    row = get_catalog_row(file_contains)
    if column not in row["columns"]:
        return pd.DataFrame(columns=["value", "n"])
    counts = pd.Series(dtype="int64")
    for chunk in iter_csv_chunks(row, columns=[column], chunksize=chunksize):
        current = chunk[column].astype(str).str.strip().value_counts(dropna=False)
        counts = counts.add(current, fill_value=0).astype("int64")
    return counts.sort_values(ascending=False).head(int(limit)).rename_axis("value").reset_index(name="n")


def key_profile(file_contains: str, key_columns: list[str], chunksize=200_000):
    row = get_catalog_row(file_contains)
    key_columns = existing_columns(row, key_columns)
    if not key_columns:
        return pd.DataFrame()

    counts = pd.Series(dtype="int64")
    n_rows = 0
    for chunk in iter_csv_chunks(row, columns=key_columns, chunksize=chunksize):
        n_rows += len(chunk)
        keys = chunk[key_columns].astype(str).apply(lambda col: col.str.strip())
        current = keys.value_counts(dropna=False)
        counts = current.astype("int64") if counts.empty else counts.add(current, fill_value=0).astype("int64")

    duplicated = counts[counts > 1]
    return pd.DataFrame(
        [
            {
                "file": row["file"],
                "key_columns": ", ".join(key_columns),
                "n_rows": n_rows,
                "n_distinct_keys": len(counts),
                "n_duplicated_keys": len(duplicated),
                "rows_in_duplicated_keys": int(duplicated.sum()) if len(duplicated) else 0,
            }
        ]
    )


def columns_matching(file_contains: str, pattern: str):
    row = get_catalog_row(file_contains)
    return [col for col in row["columns"] if re.search(pattern, normalize_name(col))]

In [9]:
import plotly.express as px


def collect_columns(file_contains: str, columns: list[str], sample_limit=None, chunksize=200_000, strict=False):
    row = get_catalog_row(file_contains)
    missing = [col for col in columns if col not in row["columns"]]
    columns = existing_columns(row, columns)
    if missing and strict:
        raise ValueError(f"Colunas ausentes: {missing}")
    if not columns:
        return pd.DataFrame()

    frames = []
    rows_seen = 0
    for chunk in iter_csv_chunks(row, columns=columns, chunksize=chunksize):
        if sample_limit is not None:
            remaining = int(sample_limit) - rows_seen
            if remaining <= 0:
                break
            chunk = chunk.head(remaining)
        rows_seen += len(chunk)
        frames.append(chunk)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=columns)


def to_numeric_series(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series.astype(str).str.strip().str.replace(",", ".", regex=False), errors="coerce")


def numeric_profile(file_contains: str, columns: list[str], sample_limit=None):
    df = collect_columns(file_contains, columns, sample_limit=sample_limit)
    if df.empty:
        return pd.DataFrame()
    numeric_df = df.apply(to_numeric_series)
    profile = numeric_df.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
    profile["missing_pct"] = numeric_df.isna().mean()
    return profile.reset_index(names="column")


def plot_numeric_distribution(file_contains: str, column: str, group_column: str | None = None, sample_limit=100_000, nbins=40):
    columns = [column] + ([group_column] if group_column else [])
    df = collect_columns(file_contains, columns, sample_limit=sample_limit)
    if df.empty or column not in df.columns:
        return None
    df[column] = to_numeric_series(df[column])
    df = df.dropna(subset=[column])
    return px.histogram(df, x=column, color=group_column, nbins=nbins, opacity=0.65, title=f"Distribuição de {column}")


def plot_value_counts(file_contains: str, column: str, limit=20):
    counts = value_counts(file_contains, column, limit=limit).sort_values("n")
    if counts.empty:
        return None
    return px.bar(counts, x="n", y="value", orientation="h", title=f"Frequência de {column}")


def municipality_key_summary(file_contains: str, column="CO_MUNICIPIO", chunksize=200_000):
    row = get_catalog_row(file_contains)
    if column not in row["columns"]:
        return pd.DataFrame()
    n_rows = 0
    codes = []
    for chunk in iter_csv_chunks(row, columns=[column], chunksize=chunksize):
        series = chunk[column].astype(str).str.strip()
        series = series[series != ""]
        n_rows += len(series)
        codes.extend(series.tolist())
    codes = pd.Series(codes, dtype="string")
    return pd.DataFrame([{"file": row["file"], "n_rows": n_rows, "n_municipios": codes.nunique(), "min_len": codes.str.len().min(), "max_len": codes.str.len().max(), "min_code": codes.min(), "max_code": codes.max()}])

## SAEB

In [10]:
show("Inventário", files_for_group("saeb")[["year", "file", "encoding", "delimiter", "n_rows", "n_cols", "size_mb"]])
show("Mudança de schema", compare_columns(BASE_GROUPS["saeb"]).head(80))

saeb_file = "saeb_2023/saeb_resultados"
saeb_core_cols = ["ANO_SAEB", "CO_UF", "NO_UF", "CO_MUNICIPIO", "NO_MUNICIPIO", "DEPENDENCIA_ADM", "LOCALIZACAO", "MEDIA_5_LP", "MEDIA_5_MT", "MEDIA_9_LP", "MEDIA_9_MT", "MEDIA_12_LP", "MEDIA_12_MT"]
show("Amostra", sample_file(saeb_file, limit=8, columns=saeb_core_cols))
show("Nulos e cardinalidade", profile_columns(saeb_file, columns=saeb_core_cols))
show("Chave candidata", key_profile(saeb_file, ["ANO_SAEB", "CO_MUNICIPIO", "DEPENDENCIA_ADM", "LOCALIZACAO"]))
show("Categorias: dependência", value_counts(saeb_file, "DEPENDENCIA_ADM"))
show("Perfil numérico", numeric_profile(saeb_file, ["MEDIA_5_LP", "MEDIA_5_MT", "MEDIA_9_LP", "MEDIA_9_MT", "MEDIA_12_LP", "MEDIA_12_MT"]))

**Inventário**

,year,file,encoding,delimiter,n_rows,n_cols,size_mb
15,2019.0,data/raw/saeb_2019/saeb_resultados_municipios_2019.csv.gz,utf-8-sig,;,68742.0,72,2.432372
16,2021.0,data/raw/saeb_2021/saeb_resultados_municipios_2021.csv.gz,utf-8-sig,;,64693.0,113,2.278586
17,2023.0,data/raw/saeb_2023/saeb_resultados_municipios_2023.csv.gz,utf-8-sig,;,69053.0,113,2.821570


**Mudança de schema**

file,column_norm,data/raw/saeb_2019/saeb_resultados_municipios_2019.csv.gz,data/raw/saeb_2021/saeb_resultados_municipios_2021.csv.gz,data/raw/saeb_2023/saeb_resultados_municipios_2023.csv.gz,n_files
0,ano_saeb,0,1,1,2
12,nivel_0_lp13,0,1,1,2
13,nivel_0_lp14,0,1,1,2
17,nivel_0_mt13,0,1,1,2
18,nivel_0_mt14,0,1,1,2
...,...,...,...,...,...
53,nivel_3_mt5,1,1,1,3
54,nivel_3_mt9,1,1,1,3
55,nivel_4_lp12,1,1,1,3
58,nivel_4_lp5,1,1,1,3


**Amostra**

...

,file,key_columns,n_rows,n_distinct_keys,n_duplicated_keys,rows_in_duplicated_keys
0,data/raw/saeb_2023/saeb_resultados_municipios_2023.csv.gz,"ANO_SAEB, CO_MUNICIPIO, DEPENDENCIA_ADM, LOCALIZACAO",69053,69053,0,0


**Categorias: dependência**

,value,n
0,"Total - Federal, Estadual e Municipal",14325
1,Total - Estadual e Municipal,14321
2,"Total - Federal, Estadual, Municipal e Privada",14317
3,Municipal,13721
4,Estadual,12054
5,Federal,315


**Perfil numérico**

,column,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max,missing_pct
0,MEDIA_5_LP,57717.0,206.227428,23.300354,122.86,155.4900,168.79,189.05,207.55,222.31,241.2120,261.0524,329.95,0.164164
1,MEDIA_5_MT,57717.0,218.317546,26.207254,124.27,163.9416,177.31,199.07,218.92,236.07,259.0000,283.8900,356.85,0.164164
2,MEDIA_9_LP,56790.0,250.823237,18.404449,170.45,207.5200,220.65,238.30,251.53,263.51,278.6300,293.4100,362.02,0.177588
3,MEDIA_9_MT,56790.0,250.630934,20.917357,172.34,207.6800,219.19,236.13,249.96,263.40,282.9355,305.0700,404.89,0.177588
4,MEDIA_12_LP,43770.0,267.946795,17.692811,189.70,224.4200,237.72,256.29,268.96,280.09,295.3400,307.1000,353.25,0.366139
5,MEDIA_12_MT,43770.0,266.236808,17.269230,203.87,229.8700,239.54,254.88,265.42,276.72,295.0100,312.1482,414.56,0.366139


In [11]:
plot_numeric_distribution(saeb_file, "MEDIA_9_MT", group_column="DEPENDENCIA_ADM")

### Decisão sobre indicadores de proficiência do SAEB

A staging do SAEB deve preservar as colunas `MEDIA_*` e `nivel_*`, além dos recortes de identificação (`ANO_SAEB`, `ID`, `CO_UF`, `NO_UF`, `CO_MUNICIPIO`, `NO_MUNICIPIO`, `DEPENDENCIA_ADM`, `LOCALIZACAO` e `CAPITAL`).

As colunas `MEDIA_*` são médias de proficiência na escala SAEB, não notas de 0 a 10. Para perguntas relacionadas ao Novo Ensino Médio, as médias com sufixos `12`, `13` e `14` são as mais relevantes porque representam o ensino médio tradicional, integrado e agregado. As médias de 5º e 9º ano devem ser preservadas como contexto da trajetória educacional do município.

As colunas `nivel_*` representam o percentual de alunos em cada faixa oficial de proficiência. Elas permitem analisar a distribuição do desempenho, e não apenas a média. Isso ajuda a responder se, no antes/depois, houve redução da concentração nos níveis baixos ou aumento da concentração nos níveis altos.

Para o ensino médio, os principais sufixos são:

- `LP12` e `MT12`: 3ª/4ª série do ensino médio tradicional;
- `LP13` e `MT13`: 3ª/4ª série do ensino médio integrado;
- `LP14` e `MT14`: 3ª/4ª série do ensino médio tradicional ou integrado, de forma agregada.

As faixas de Língua Portuguesa no ensino médio (`nivel_*_LP12`, `nivel_*_LP13` e `nivel_*_LP14`) vão de `nivel_0` a `nivel_8`:

- `nivel_0`: desempenho menor que 225;
- `nivel_1`: desempenho maior ou igual a 225 e menor que 250;
- `nivel_2`: desempenho maior ou igual a 250 e menor que 275;
- `nivel_3`: desempenho maior ou igual a 275 e menor que 300;
- `nivel_4`: desempenho maior ou igual a 300 e menor que 325;
- `nivel_5`: desempenho maior ou igual a 325 e menor que 350;
- `nivel_6`: desempenho maior ou igual a 350 e menor que 375;
- `nivel_7`: desempenho maior ou igual a 375 e menor que 400;
- `nivel_8`: desempenho maior ou igual a 400.

As faixas de Matemática no ensino médio (`nivel_*_MT12`, `nivel_*_MT13` e `nivel_*_MT14`) vão de `nivel_0` a `nivel_10`:

- `nivel_0`: desempenho menor que 225;
- `nivel_1`: desempenho maior ou igual a 225 e menor que 250;
- `nivel_2`: desempenho maior ou igual a 250 e menor que 275;
- `nivel_3`: desempenho maior ou igual a 275 e menor que 300;
- `nivel_4`: desempenho maior ou igual a 300 e menor que 325;
- `nivel_5`: desempenho maior ou igual a 325 e menor que 350;
- `nivel_6`: desempenho maior ou igual a 350 e menor que 375;
- `nivel_7`: desempenho maior ou igual a 375 e menor que 400;
- `nivel_8`: desempenho maior ou igual a 400 e menor que 425;
- `nivel_9`: desempenho maior ou igual a 425 e menor que 450;
- `nivel_10`: desempenho maior ou igual a 450.


## Censo Escolar 2019-2024

In [12]:
show("Inventário", files_for_group("censo_escolar_2019_2024")[["year", "file", "encoding", "delimiter", "n_rows", "n_cols", "size_mb"]])
show("Mudança de schema", compare_columns(BASE_GROUPS["censo_escolar_2019_2024"]).head(120))

censo_2024_file = "censo_escolar_2024/microdados_ed_basica_2024"
censo_core_cols = ["NU_ANO_CENSO", "CO_UF", "SG_UF", "CO_MUNICIPIO", "NO_MUNICIPIO", "CO_ENTIDADE", "NO_ENTIDADE", "TP_DEPENDENCIA", "TP_LOCALIZACAO", "IN_MEDIACAO_PRESENCIAL", "IN_MEDIACAO_SEMIPRESENCIAL", "IN_MEDIACAO_EAD", "QT_MAT_MED", "QT_MAT_MED_PROP", "QT_MAT_MED_INT", "QT_MAT_MED_NM"]
show("Amostra 2024", sample_file(censo_2024_file, limit=8, columns=censo_core_cols))
show("Nulos e cardinalidade 2024", profile_columns(censo_2024_file, columns=censo_core_cols))
show("Chave candidata 2024", key_profile(censo_2024_file, ["NU_ANO_CENSO", "CO_ENTIDADE"]))
show("Categorias: dependência 2024", value_counts(censo_2024_file, "TP_DEPENDENCIA"))
show("Perfil numérico 2024", numeric_profile(censo_2024_file, ["QT_MAT_MED", "QT_MAT_MED_PROP", "QT_MAT_MED_INT", "QT_MAT_MED_NM"]))

**Inventário**

,year,file,encoding,delimiter,n_rows,n_cols,size_mb
0,2019.0,data/raw/censo_escolar_2019/microdados_ed_basica_2019.csv.gz,cp1252,;,228521.0,370,26.923068
1,2020.0,data/raw/censo_escolar_2020/microdados_ed_basica_2020.csv.gz,cp1252,;,224229.0,370,26.785557
2,2021.0,data/raw/censo_escolar_2021/microdados_ed_basica_2021.csv.gz,cp1252,;,221140.0,370,26.710618
3,2022.0,data/raw/censo_escolar_2022/microdados_ed_basica_2022.csv.gz,cp1252,;,224649.0,385,27.616155
4,2023.0,data/raw/censo_escolar_2023/microdados_ed_basica_2023.csv.gz,cp1252,;,217625.0,408,32.685275
6,2024.0,data/raw/censo_escolar_2024/microdados_ed_basica_2024.csv.gz,cp1252,;,215545.0,426,31.016352


**Mudança de schema**

file,column_norm,data/raw/censo_escolar_2019/microdados_ed_basica_2019.csv.gz,data/raw/censo_escolar_2020/microdados_ed_basica_2020.csv.gz,data/raw/censo_escolar_2021/microdados_ed_basica_2021.csv.gz,data/raw/censo_escolar_2022/microdados_ed_basica_2022.csv.gz,data/raw/censo_escolar_2023/microdados_ed_basica_2023.csv.gz,data/raw/censo_escolar_2024/microdados_ed_basica_2024.csv.gz,n_files
29,in_acessibilidade_sinalizacao,0,0,0,0,0,1,1
33,in_agua_carro_pipa,0,0,0,0,0,1,1
42,in_area_plantio,0,0,0,0,0,1,1
70,in_educ_amb_conteudo,0,0,0,0,0,1,1
71,in_educ_amb_curricular,0,0,0,0,0,1,1
...,...,...,...,...,...,...,...,...
90,in_equip_foto,1,1,1,1,0,0,4
97,in_equip_retroprojetor,1,1,1,1,0,0,4
101,in_equip_videocassete,1,1,1,1,0,0,4
114,in_final_semana,1,1,1,1,0,0,4


**Amostra 2024**

...

,file,key_columns,n_rows,n_distinct_keys,n_duplicated_keys,rows_in_duplicated_keys
0,data/raw/censo_escolar_2024/microdados_ed_basica_2024.csv.gz,"NU_ANO_CENSO, CO_ENTIDADE",215545,215545,0,0


**Categorias: dependência 2024**

,value,n
0,3,128999
1,4,52568
2,2,33254
3,1,724


**Perfil numérico 2024**

,column,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max,missing_pct
0,QT_MAT_MED,179286.0,43.452339,146.990743,0.0,0.0,0.0,0.0,0.0,0.0,305.75,741.0,3341.0,0.16822
1,QT_MAT_MED_PROP,179286.0,37.744732,134.890901,0.0,0.0,0.0,0.0,0.0,0.0,256.00,693.0,3116.0,0.16822
2,QT_MAT_MED_NM,179286.0,0.208443,7.403855,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,1393.0,0.16822
3,QT_MAT_MED_INT,179286.0,9.410807,52.985097,0.0,0.0,0.0,0.0,0.0,0.0,25.00,295.0,1776.0,0.16822


In [13]:
plot_numeric_distribution(censo_2024_file, "QT_MAT_MED", sample_limit=100_000)

## Censo Escolar 2025

In [14]:
show("Inventário", files_for_group("censo_escolar_2025")[["file", "encoding", "delimiter", "n_rows", "n_cols", "size_mb"]])
show("Colunas entre tabelas", compare_columns(BASE_GROUPS["censo_escolar_2025"]).head(140))

censo_2025_keys = {
    "censo_escolar_2025/Tabela_Escola_2025": ["NU_ANO_CENSO", "CO_ENTIDADE"],
    "censo_escolar_2025/Tabela_Matricula_2025": ["NU_ANO_CENSO", "CO_ENTIDADE"],
    "censo_escolar_2025/Tabela_Turma_2025": ["NU_ANO_CENSO", "CO_ENTIDADE"],
    "censo_escolar_2025/Tabela_Docente_2025": ["NU_ANO_CENSO", "CO_ENTIDADE"],
    "censo_escolar_2025/Tabela_Gestor_Escolar_2025": ["NU_ANO_CENSO", "CO_ENTIDADE"],
    "censo_escolar_2025/Tabela_Curso_Tecnico_2025": ["NU_ANO_CENSO", "CO_ENTIDADE"],
}
show("Amostra: escola", sample_file("censo_escolar_2025/Tabela_Escola_2025", limit=8, columns=["NU_ANO_CENSO", "CO_UF", "SG_UF", "CO_MUNICIPIO", "NO_MUNICIPIO", "CO_ENTIDADE", "NO_ENTIDADE", "TP_DEPENDENCIA", "TP_LOCALIZACAO"]))
show("Chaves candidatas", pd.concat([key_profile(file, keys) for file, keys in censo_2025_keys.items()], ignore_index=True))
show("Perfil matrículas", numeric_profile("censo_escolar_2025/Tabela_Matricula_2025", ["QT_MAT_MED", "QT_MAT_MED_PROP", "QT_MAT_MED_INT", "QT_MAT_MED_NM"]))

**Inventário**

,file,encoding,delimiter,n_rows,n_cols,size_mb
8,data/raw/censo_escolar_2025/Tabela_Curso_Tecnico_2025.csv.gz,cp1252,;,32136.0,20,0.307333
9,data/raw/censo_escolar_2025/Tabela_Docente_2025.csv.gz,utf-8-sig,;,178772.0,156,8.565481
10,data/raw/censo_escolar_2025/Tabela_Escola_2025.csv.gz,cp1252,;,214192.0,302,20.279098
11,data/raw/censo_escolar_2025/Tabela_Gestor_Escolar_2025.csv.gz,utf-8-sig,;,180540.0,65,1.780763
12,data/raw/censo_escolar_2025/Tabela_Matricula_2025.csv.gz,utf-8-sig,;,178766.0,237,13.863099
13,data/raw/censo_escolar_2025/Tabela_Turma_2025.csv.gz,utf-8-sig,;,178772.0,190,6.153510


**Colunas entre tabelas**

file,column_norm,data/raw/censo_escolar_2025/Tabela_Curso_Tecnico_2025.csv.gz,data/raw/censo_escolar_2025/Tabela_Docente_2025.csv.gz,data/raw/censo_escolar_2025/Tabela_Escola_2025.csv.gz,data/raw/censo_escolar_2025/Tabela_Gestor_Escolar_2025.csv.gz,data/raw/censo_escolar_2025/Tabela_Matricula_2025.csv.gz,data/raw/censo_escolar_2025/Tabela_Turma_2025.csv.gz,n_files
0,co_cep,0,0,1,0,0,0,1
1,co_curso_educ_profissional,1,0,0,0,0,0,1
2,co_distrito,0,0,1,0,0,0,1
4,co_escola_sede_vinculada,0,0,1,0,0,0,1
5,co_ies_ofertante,0,0,1,0,0,0,1
...,...,...,...,...,...,...,...,...
136,in_forma_cont_prestacao_serv,0,0,1,0,0,0,1
137,in_forma_cont_termo_colabora,0,0,1,0,0,0,1
138,in_forma_cont_termo_fomento,0,0,1,0,0,0,1
139,in_internet,0,0,1,0,0,0,1


**Amostra: escola**

,NU_ANO_CENSO,SG_UF,CO_UF,NO_MUNICIPIO,CO_MUNICIPIO,NO_ENTIDADE,CO_ENTIDADE,TP_DEPENDENCIA,TP_LOCALIZACAO
0,2025,RO,11,Alta Floresta D'Oeste,1100015,EIEEF HAP BITT TUPARI,11022558,2,2
1,2025,RO,11,Alta Floresta D'Oeste,1100015,CEEJA LUIZ VAZ DE CAMOES,11024275,2,1
2,2025,RO,11,Alta Floresta D'Oeste,1100015,EMMEF 7 DE SETEMBRO,11024291,3,2
3,2025,RO,11,Alta Floresta D'Oeste,1100015,EMEIEF BOA ESPERANCA,11024666,3,2
4,2025,RO,11,Alta Floresta D'Oeste,1100015,EEEFM EURIDICE LOPES PEDROSO,11024682,2,1
5,2025,RO,11,Alta Floresta D'Oeste,1100015,EMEIEF JOSE BASILIO DA GAMA,11024917,3,2
6,2025,RO,11,Alta Floresta D'Oeste,1100015,EEEMTI JUSCELINO KUBITSCHEK DE OLIVEIRA,11024968,2,1
7,2025,RO,11,Alta Floresta D'Oeste,1100015,EMMEF MALBA TAHAN,11025034,3,2


**Chaves candidatas**

,file,key_columns,n_rows,n_distinct_keys,n_duplicated_keys,rows_in_duplicated_keys
0,data/raw/censo_escolar_2025/Tabela_Escola_2025.csv.gz,"NU_ANO_CENSO, CO_ENTIDADE",214192,214192,0,0
1,data/raw/censo_escolar_2025/Tabela_Matricula_2025.csv.gz,"NU_ANO_CENSO, CO_ENTIDADE",178766,178766,0,0
2,data/raw/censo_escolar_2025/Tabela_Turma_2025.csv.gz,"NU_ANO_CENSO, CO_ENTIDADE",178772,178772,0,0
3,data/raw/censo_escolar_2025/Tabela_Docente_2025.csv.gz,"NU_ANO_CENSO, CO_ENTIDADE",178772,178772,0,0
4,data/raw/censo_escolar_2025/Tabela_Gestor_Escolar_2025.csv.gz,"NU_ANO_CENSO, CO_ENTIDADE",180540,180540,0,0
5,data/raw/censo_escolar_2025/Tabela_Curso_Tecnico_2025.csv.gz,"NU_ANO_CENSO, CO_ENTIDADE",32136,12026,6923,27033


**Perfil matrículas**

,column,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max,missing_pct
0,QT_MAT_MED,178766.0,41.231996,137.318167,0.0,0.0,0.0,0.0,0.0,0.0,294.0,681.35,4444.0,0.0
1,QT_MAT_MED_PROP,178766.0,31.439547,116.261061,0.0,0.0,0.0,0.0,0.0,0.0,213.0,588.00,4328.0,0.0
2,QT_MAT_MED_NM,178766.0,0.181964,7.042502,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,1378.0,0.0
3,QT_MAT_MED_INT,178766.0,10.123497,54.207180,0.0,0.0,0.0,0.0,0.0,0.0,38.0,303.00,1767.0,0.0


## IBGE e seeds

In [15]:
show("Raw IBGE", files_for_group("ibge_raw")[["file", "encoding", "delimiter", "n_rows", "n_cols", "size_mb"]])
show("Seeds", files_for_group("seeds")[["file", "encoding", "delimiter", "n_cols", "size_mb"]])

ibge_file = "br_ibge_censo_2022_alfabetizacao"
show("Amostra IBGE", sample_file(ibge_file, limit=8))
show("Nulos e cardinalidade IBGE", profile_columns(ibge_file, columns=["id_municipio", "cor_raca", "sexo", "grupo_idade", "alfabetizacao", "populacao"]))
show("Categorias: alfabetização", value_counts(ibge_file, "alfabetizacao"))
show("Perfil numérico IBGE", numeric_profile(ibge_file, ["populacao"]))

**Raw IBGE**

,file,encoding,delimiter,n_rows,n_cols,size_mb
14,data/raw/br_ibge_censo_2022_alfabetizacao_grupo_idade_sexo_raca.csv.gz,utf-8-sig,",",779800.0,6,4.097044


**Seeds**

,file,encoding,delimiter,n_cols,size_mb
18,seeds/br_bd_diretorios_brasil_municipio.csv,utf-8-sig,",",27,1.297774
19,seeds/br_ibge_censo_2022_municipio.csv,utf-8-sig,",",13,0.311889
20,seeds/ibge_pib_municipios_clean.csv,utf-8-sig,;,7,0.325558


**Amostra IBGE**

...

,column,n_rows,n_missing,missing_pct,n_distinct,min_value,max_value
0,id_municipio,779800,0,0.000000,5570,1100015,5300108
1,cor_raca,779800,0,0.000000,5,Amarela,Preta
2,sexo,779800,0,0.000000,2,Homens,Mulheres
3,grupo_idade,779800,0,0.000000,7,15 a 19 anos,65 anos ou mais
4,alfabetizacao,779800,0,0.000000,2,Alfabetizadas,Não alfabetizadas
5,populacao,779800,250085,0.320704,8281,1,9997


**Categorias: alfabetização**

,value,n
0,Alfabetizadas,389900
1,Não alfabetizadas,389900


**Perfil numérico IBGE**

,column,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max,missing_pct
0,populacao,529715.0,307.603172,3154.010564,1.0,1.0,1.0,5.0,30.0,139.0,924.0,4276.86,536242.0,0.320704


In [ ]:
ibge_sample_file = RAW_DIR / "br_ibge_censo_2022_alfabetizacao_grupo_idade_sexo_raca.csv.gz"

# `populacao` é a contagem de pessoas em cada combinação de município e recortes demográficos.
# Neste arquivo, ela vem junto de `id_municipio`, `cor_raca`, `sexo`, `grupo_idade` e `alfabetizacao`.
con.sql(
    f"""
    select
        id_municipio,
        cor_raca,
        sexo,
        grupo_idade,
        alfabetizacao,
        populacao
    from read_csv_auto('{ibge_sample_file.as_posix()}')
    where populacao is not null
    limit 5
    """
).df()

,id_municipio,cor_raca,sexo,grupo_idade,alfabetizacao,populacao
0,1100023,Indígena,Homens,15 a 19 anos,Não alfabetizadas,None
1,1100262,Amarela,Mulheres,15 a 19 anos,Não alfabetizadas,None
2,1101005,Amarela,Mulheres,15 a 19 anos,Não alfabetizadas,None
3,1101435,Amarela,Mulheres,15 a 19 anos,Não alfabetizadas,None
4,1101435,Indígena,Mulheres,15 a 19 anos,Não alfabetizadas,None


## Chaves territoriais

In [16]:
pd.concat(
    [
        municipality_key_summary("saeb_2023/saeb_resultados").assign(base="saeb_2023"),
        municipality_key_summary("censo_escolar_2024/microdados_ed_basica_2024").assign(base="censo_2024"),
        municipality_key_summary("censo_escolar_2025/Tabela_Escola_2025").assign(base="censo_2025_escola"),
    ],
    ignore_index=True,
)

,file,n_rows,n_municipios,min_len,max_len,min_code,max_code,base
0,data/raw/saeb_2023/saeb_resultados_municipios_2023.csv.gz,69053,5557,7,7,1100015,5300108,saeb_2023
1,data/raw/censo_escolar_2024/microdados_ed_basica_2024.csv.gz,215545,5570,7,7,1100015,5300108,censo_2024
2,data/raw/censo_escolar_2025/Tabela_Escola_2025.csv.gz,214192,5571,7,7,1100015,5300108,censo_2025_escola


## Código real x mascarado

In [17]:
def find_header_row(raw: pd.DataFrame, required_terms: list[str]) -> int | None:
    required = {normalize_name(term) for term in required_terms}
    for idx, row in raw.iterrows():
        values = {normalize_name(value) for value in row.dropna().astype(str)}
        if required.issubset(values):
            return int(idx)
    return None


def first_existing_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    normalized = {normalize_name(col): col for col in df.columns}
    for candidate in candidates:
        found = normalized.get(normalize_name(candidate))
        if found is not None:
            return found
    return None


def load_saeb_dictionary(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep=";", encoding="utf-8-sig", skiprows=1, dtype=str)
    variable_col = first_existing_column(df, ["Variável", "Variavel", "Nome da Variável"])
    description_col = first_existing_column(df, ["Descrição", "Descricao", "Descrição da Variável"])
    result = df[[variable_col, description_col]].copy()
    result.columns = ["column", "description"]
    result["dictionary_file"] = path.name
    result["dictionary_sheet"] = None
    result["dictionary_family"] = "saeb"
    return result.dropna(subset=["column"])


def load_censo_dictionary(path: Path) -> pd.DataFrame:
    frames = []
    for sheet_name, raw in pd.read_excel(path, sheet_name=None, header=None, dtype=str).items():
        header_idx = find_header_row(raw, ["Nome da Variável", "Descrição da Variável"])
        if header_idx is None:
            continue
        df = raw.iloc[header_idx + 1 :].copy()
        df.columns = raw.iloc[header_idx].tolist()
        variable_col = first_existing_column(df, ["Nome da Variável", "Variável", "Variavel"])
        description_col = first_existing_column(df, ["Descrição da Variável", "Descrição", "Descricao"])
        if variable_col is None or description_col is None:
            continue
        selected = df[[variable_col, description_col]].copy()
        selected.columns = ["column", "description"]
        selected["dictionary_file"] = path.name
        selected["dictionary_sheet"] = sheet_name
        selected["dictionary_family"] = "censo_escolar"
        frames.append(selected)
    if not frames:
        return pd.DataFrame(columns=["column", "description", "dictionary_file", "dictionary_sheet", "dictionary_family"])
    result = pd.concat(frames, ignore_index=True).dropna(subset=["column"])
    return result[result["column"].astype(str).str.match(r"^[A-Za-z_][A-Za-z0-9_]*$", na=False)]


def load_available_dictionaries() -> pd.DataFrame:
    frames = []
    saeb_path = ROOT / "docs" / "Dicionario_Resultados_Saeb_2023.csv"
    if saeb_path.exists():
        frames.append(load_saeb_dictionary(saeb_path))
    for censo_path in sorted((ROOT / "docs").glob("Dicionario_Censo_Escolar_*.xlsx")):
        try:
            frames.append(load_censo_dictionary(censo_path))
        except Exception as exc:
            print(f"Aviso: {censo_path.name}: {exc}")
    dictionary = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    if dictionary.empty:
        return dictionary
    dictionary["column_norm"] = dictionary["column"].map(normalize_name)
    dictionary["description_norm"] = dictionary["description"].fillna("").map(normalize_name)
    return dictionary.drop_duplicates(["dictionary_file", "dictionary_sheet", "column_norm"])


dictionary_catalog = load_available_dictionaries()
dictionary_catalog[["dictionary_family", "dictionary_file", "dictionary_sheet", "column", "description"]].head(20)

,dictionary_family,dictionary_file,dictionary_sheet,column,description
0,saeb,Dicionario_Resultados_Saeb_2023.csv,None,ANO_SAEB,Ano de aplicação do Saeb
1,saeb,Dicionario_Resultados_Saeb_2023.csv,None,ID,nível de agragação do Resultado
2,saeb,Dicionario_Resultados_Saeb_2023.csv,None,CO_UF,Código da Unidade da Federação
3,saeb,Dicionario_Resultados_Saeb_2023.csv,None,NO_UF,Nome da Unidade da Federação
4,saeb,Dicionario_Resultados_Saeb_2023.csv,None,CO_MUNICIPIO,Código do Municipio
5,saeb,Dicionario_Resultados_Saeb_2023.csv,None,NO_MUNICIPIO,Nome do Município
6,saeb,Dicionario_Resultados_Saeb_2023.csv,None,DEPENDENCIA_ADM,Dependência administrativa
7,saeb,Dicionario_Resultados_Saeb_2023.csv,None,LOCALIZACAO,Indicador de resultados referente a alunos da localização urbana ou rural
8,saeb,Dicionario_Resultados_Saeb_2023.csv,None,CAPITAL,Indicador de resultados referente a alunos da Capital ou do Interior
9,saeb,Dicionario_Resultados_Saeb_2023.csv,None,PC_ALFABETIZADO,Percentual de alunos alfabetizados


In [18]:
MASK_TERMS = r"mascar|anonim|sigilo|identificador interno|codigo interno|id anonim"
REAL_CODE_TERMS = r"codigo do municipio|codigo da unidade da federacao|codigo da escola|codigo da regiao|codigo da mesorregiao|codigo da microrregiao"
INTEGRATION_COLUMN_RULES = {
    "co_municipio": "codigo_real_validavel",
    "co_uf": "codigo_real_validavel",
    "sg_uf": "codigo_real_validavel",
    "id_municipio": "codigo_real_validavel",
    "co_entidade": "aceito_pelo_dicionario",
    "co_regiao": "aceito_pelo_dicionario",
    "co_mesorregiao": "aceito_pelo_dicionario",
    "co_microrregiao": "aceito_pelo_dicionario",
    "co_regiao_geog_interm": "aceito_pelo_dicionario",
    "co_regiao_geog_imed": "aceito_pelo_dicionario",
}


def classify_identifier_from_metadata(row) -> tuple[str, str]:
    column_norm = row["column_norm"]
    description_norm = normalize_name(row.get("description", ""))
    family = row.get("family", "")

    if re.search(MASK_TERMS, description_norm):
        return "codigo_mascarado_pelo_dicionario", "dicionário cita mascaramento/anonimização"
    if column_norm in INTEGRATION_COLUMN_RULES:
        status = INTEGRATION_COLUMN_RULES[column_norm]
        evidence = "regra por nome de coluna"
        if re.search(REAL_CODE_TERMS, description_norm):
            evidence += "; dicionário descreve código oficial"
        return status, evidence
    if family == "saeb" and re.search(r"(^id_|aluno|turma|escola|questionario|prova)", column_norm):
        return "provavel_codigo_mascarado_ou_interno", "identificador sensível em família SAEB"
    if re.search(r"(^co_|^id_|codigo|cod_)", column_norm) or re.search(r"codigo", description_norm):
        return "codigo_indeterminado", "parece código, mas falta validação externa"
    return "nao_e_codigo_de_integracao", "sem padrão de chave de integração"


def classify_identifier_columns(schema_df: pd.DataFrame, dictionary_df: pd.DataFrame) -> pd.DataFrame:
    dictionary_lookup = dictionary_df.drop_duplicates("column_norm")[["column_norm", "description", "dictionary_file", "dictionary_sheet", "dictionary_family"]]
    candidates = schema_df[
        schema_df["column_norm"].str.contains(r"(?:^co_|^id_|codigo|cod_|municipio|uf|entidade|escola|regiao|micro|meso|aluno|turma)", regex=True, na=False)
    ].merge(dictionary_lookup, on="column_norm", how="left")
    classified = candidates.copy()
    classified[["codigo_status", "evidencia"]] = classified.apply(lambda row: pd.Series(classify_identifier_from_metadata(row)), axis=1)
    return classified[["source_layer", "family", "year", "file", "column", "description", "dictionary_file", "codigo_status", "evidencia"]]


code_classification = classify_identifier_columns(all_schema_catalog, dictionary_catalog)
code_classification.sort_values(["family", "file", "codigo_status", "column"]).head(120)

,source_layer,family,year,file,column,description,dictionary_file,codigo_status,evidencia
12,raw,censo_escolar,2019.0,data/raw/censo_escolar_2019/microdados_ed_basica_2019.csv.gz,CO_ENTIDADE,Código da Escola,Dicionario_Censo_Escolar_2019.xlsx,aceito_pelo_dicionario,regra por nome de coluna
8,raw,censo_escolar,2019.0,data/raw/censo_escolar_2019/microdados_ed_basica_2019.csv.gz,CO_MESORREGIAO,Código da Mesorregião,Dicionario_Censo_Escolar_2019.xlsx,aceito_pelo_dicionario,regra por nome de coluna
10,raw,censo_escolar,2019.0,data/raw/censo_escolar_2019/microdados_ed_basica_2019.csv.gz,CO_MICRORREGIAO,Código da Microrregião,Dicionario_Censo_Escolar_2019.xlsx,aceito_pelo_dicionario,regra por nome de coluna
1,raw,censo_escolar,2019.0,data/raw/censo_escolar_2019/microdados_ed_basica_2019.csv.gz,CO_REGIAO,Código da região geográfica,Dicionario_Censo_Escolar_2019.xlsx,aceito_pelo_dicionario,regra por nome de coluna
15,raw,censo_escolar,2019.0,data/raw/censo_escolar_2019/microdados_ed_basica_2019.csv.gz,CO_CEP,CEP,Dicionario_Censo_Escolar_2019.xlsx,codigo_indeterminado,"parece código, mas falta validação externa"
...,...,...,...,...,...,...,...,...,...
122,raw,censo_escolar,2021.0,data/raw/censo_escolar_2021/microdados_ed_basica_2021.csv.gz,IN_TABLET_ALUNO,Computadores em uso pelos alunos - Tablet,Dicionario_Censo_Escolar_2019.xlsx,nao_e_codigo_de_integracao,sem padrão de chave de integração
99,raw,censo_escolar,2021.0,data/raw/censo_escolar_2021/microdados_ed_basica_2021.csv.gz,NO_ENTIDADE,Nome da Escola,Dicionario_Censo_Escolar_2019.xlsx,nao_e_codigo_de_integracao,sem padrão de chave de integração
93,raw,censo_escolar,2021.0,data/raw/censo_escolar_2021/microdados_ed_basica_2021.csv.gz,NO_MESORREGIAO,Nome da Mesorregião,Dicionario_Censo_Escolar_2019.xlsx,nao_e_codigo_de_integracao,sem padrão de chave de integração
95,raw,censo_escolar,2021.0,data/raw/censo_escolar_2021/microdados_ed_basica_2021.csv.gz,NO_MICRORREGIAO,Nome da Microrregião,Dicionario_Censo_Escolar_2019.xlsx,nao_e_codigo_de_integracao,sem padrão de chave de integração


In [19]:
def load_official_code_sets() -> dict[str, set[str]]:
    refs = {"id_municipio": set(), "id_uf": set(), "sigla_uf": set()}
    municipio_path = SEEDS_DIR / "br_bd_diretorios_brasil_municipio.csv"
    if municipio_path.exists():
        municipio = pd.read_csv(municipio_path, dtype=str)
        refs["id_municipio"] = set(municipio["id_municipio"].dropna().astype(str).str.strip())
        refs["id_uf"] = set(municipio["id_uf"].dropna().astype(str).str.strip().str.zfill(2))
        refs["sigla_uf"] = set(municipio["sigla_uf"].dropna().astype(str).str.strip())
    return refs


def reference_for_column(column: str) -> tuple[str | None, set[str]]:
    refs = load_official_code_sets()
    column_norm = normalize_name(column)
    if column_norm in {"co_municipio", "id_municipio"}:
        return "seed municipio.id_municipio", refs["id_municipio"]
    if column_norm == "co_uf":
        return "seed municipio.id_uf", refs["id_uf"]
    if column_norm == "sg_uf":
        return "seed municipio.sigla_uf", refs["sigla_uf"]
    return None, set()


def validate_code_values(file_contains: str, columns: list[str], sample_limit=None, chunksize=200_000) -> pd.DataFrame:
    row = get_catalog_row(file_contains)
    columns = existing_columns(row, columns)
    values = {col: set() for col in columns}
    rows_seen = 0
    for chunk in iter_csv_chunks(row, columns=columns, chunksize=chunksize):
        if sample_limit is not None:
            remaining = int(sample_limit) - rows_seen
            if remaining <= 0:
                break
            chunk = chunk.head(remaining)
        rows_seen += len(chunk)
        for col in columns:
            series = chunk[col].astype(str).str.strip()
            if normalize_name(col) == "co_uf":
                series = series.str.zfill(2)
            values[col].update(series[series != ""].dropna().unique().tolist())

    rows = []
    for col, observed in values.items():
        ref_name, ref_values = reference_for_column(col)
        matched = observed & ref_values if ref_values else set()
        match_pct = len(matched) / len(observed) if observed and ref_values else None
        status = "codigo_real_validado" if match_pct == 1 else "sem_referencia_externa" if not ref_values else "codigo_parcialmente_compativel"
        rows.append({"file": row["file"], "column": col, "n_rows_lidos": rows_seen, "n_valores_observados": len(observed), "referencia": ref_name, "compatibilidade_pct": match_pct, "validacao_status": status, "exemplos_fora_referencia": sorted(observed - ref_values)[:10] if ref_values else []})
    return pd.DataFrame(rows)


pd.concat(
    [
        validate_code_values("saeb_2023/saeb_resultados", ["CO_UF", "CO_MUNICIPIO"]),
        validate_code_values("censo_escolar_2024/microdados_ed_basica_2024", ["CO_UF", "SG_UF", "CO_MUNICIPIO", "CO_ENTIDADE"]),
        validate_code_values("censo_escolar_2025/Tabela_Escola_2025", ["CO_UF", "SG_UF", "CO_MUNICIPIO", "CO_ENTIDADE"]),
        validate_code_values("br_ibge_censo_2022_alfabetizacao", ["id_municipio"]),
    ],
    ignore_index=True,
)

,file,column,n_rows_lidos,n_valores_observados,referencia,compatibilidade_pct,validacao_status,exemplos_fora_referencia
0,data/raw/saeb_2023/saeb_resultados_municipios_2023.csv.gz,CO_UF,69053,27,seed municipio.id_uf,1.0,codigo_real_validado,[]
1,data/raw/saeb_2023/saeb_resultados_municipios_2023.csv.gz,CO_MUNICIPIO,69053,5557,seed municipio.id_municipio,1.0,codigo_real_validado,[]
2,data/raw/censo_escolar_2024/microdados_ed_basica_2024.csv.gz,CO_UF,215545,27,seed municipio.id_uf,1.0,codigo_real_validado,[]
3,data/raw/censo_escolar_2024/microdados_ed_basica_2024.csv.gz,SG_UF,215545,27,seed municipio.sigla_uf,1.0,codigo_real_validado,[]
4,data/raw/censo_escolar_2024/microdados_ed_basica_2024.csv.gz,CO_MUNICIPIO,215545,5570,seed municipio.id_municipio,1.0,codigo_real_validado,[]
5,data/raw/censo_escolar_2024/microdados_ed_basica_2024.csv.gz,CO_ENTIDADE,215545,215545,None,NaN,sem_referencia_externa,[]
6,data/raw/censo_escolar_2025/Tabela_Escola_2025.csv.gz,CO_UF,214192,27,seed municipio.id_uf,1.0,codigo_real_validado,[]
7,data/raw/censo_escolar_2025/Tabela_Escola_2025.csv.gz,SG_UF,214192,27,seed municipio.sigla_uf,1.0,codigo_real_validado,[]
8,data/raw/censo_escolar_2025/Tabela_Escola_2025.csv.gz,CO_MUNICIPIO,214192,5571,seed municipio.id_municipio,1.0,codigo_real_validado,[]
9,data/raw/censo_escolar_2025/Tabela_Escola_2025.csv.gz,CO_ENTIDADE,214192,214192,None,NaN,sem_referencia_externa,[]


## Decisões para staging

- `CO_MUNICIPIO`/`id_municipio`, `CO_UF` e `SG_UF` devem ser usados quando a validação retornar `codigo_real_validado`.
- SAEB entra pela base agregada municipal; identificadores sensíveis de microdados continuam como internos/mascarados até prova externa.
- Censo Escolar 2025 fica separado por tabela no staging, porque não é continuação direta do arquivo largo de 2019-2024.
- Nulos estruturais de oferta/medição devem ser preservados; imputação fica para camadas analíticas.
- Testes dbt prioritários: `not_null`, chaves compostas por granularidade, `accepted_values`, faixas SAEB e relacionamento de município com seed oficial.

Secretários
id_municipio + ano/período de gestão
|
v
SAEB
id_municipio + ano_saeb + recortes
|
v
Censo Escolar
id_municipio + ano_censo, ou id_escola + ano_censo
|
v
PIB / IBGE / Alfabetização
id_municipio + ano